In [1]:
# ================================================================
# LIGHTGBM PARA PREDICCIÓN DE CASOS DE DENGUE
# VERSIÓN REFORZADA CONTRA SOBREAJUSTE Y DATA LEAKAGE
#
# Cambios principales:
# 1. Elimina columnas duplicadas del Excel.
# 2. Mantiene separación temporal estricta: 2021-2025 TRAIN / 2026 TEST.
# 3. La selección de variables se realiza DENTRO de cada fold.
# 4. Los umbrales de picos se calculan DENTRO de cada fold.
# 5. Se reduce el número de variables a 12.
# 6. Se elimina el esquema de pesos agresivos que favorecía el ajuste de picos.
# 7. Se usa una ponderación de picos suave y limitada.
# 8. Se restringe la complejidad de LightGBM.
# 9. Se usa validación walk-forward con 3 folds cuando los datos lo permiten.
# 10. El número de árboles final se obtiene de la mediana de los folds.
# 11. 2026 permanece completamente aislado para la evaluación final.
# 12. Se compara contra el baseline naive y se guardan resultados.
# ================================================================

import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ================================================================
# 1. CONFIGURACIÓN
# ================================================================

input_file = (
    r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina"
    r"\4_algoritmo_LightGBM\2_datos"
    r"\1_raw\2_meteo_epi_2021-2026_1_rezagos.xlsx"
)

output_dir = (
    r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina"
    r"\4_algoritmo_LightGBM\3_resultados"
)

processed_dir = (
    r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina"
    r"\4_algoritmo_LightGBM\2_datos\2_procesados"
)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

RANDOM_STATE = 42
TEST_YEAR = 2026

# Con 197 observaciones de TRAIN, 30 variables es demasiado.
MAX_SELECTED_FEATURES = 12

# Menor número de ensayos para evitar optimizar excesivamente sobre pocos folds.
N_TRIALS = 40

# Máximo de árboles.
MAX_BOOST_ROUND = 1200

# ================================================================
# 2. CARGA Y LIMPIEZA
# ================================================================

print("=" * 80)
print("CARGANDO DATOS")
print("=" * 80)

df = pd.read_excel(input_file)

if "fecha" not in df.columns:
    raise ValueError("No existe la columna 'fecha'.")

if "casos_dengue" not in df.columns:
    raise ValueError("No existe la columna 'casos_dengue'.")

df["fecha"] = pd.to_datetime(df["fecha"])

# Orden cronológico.
df = df.sort_values("fecha").reset_index(drop=True)

# ------------------------------------------------
# CORRECCIÓN IMPORTANTE:
# El archivo original permite columnas duplicadas.
# Eso explica, entre otras cosas, que RF pueda reportar dos
# variables con el mismo nombre.
# ------------------------------------------------

duplicated_columns = df.columns[df.columns.duplicated()].tolist()

if duplicated_columns:
    print("\nColumnas duplicadas eliminadas:")
    print(sorted(set(duplicated_columns)))
    df = df.loc[:, ~df.columns.duplicated()].copy()

target_col = "casos_dengue"
exclude_cols = ["fecha", "año", "semana_epi"]

if "año" not in df.columns:
    df["año"] = df["fecha"].dt.year

if "semana_epi" not in df.columns:
    df["semana_epi"] = (
        df["fecha"].dt.isocalendar().week.astype(int)
    )

predictor_cols = [
    col for col in df.columns
    if col not in exclude_cols + [target_col]
]

print(f"Registros originales: {len(df)}")
print(f"Predictores iniciales: {len(predictor_cols)}")
print(f"Periodo: {df['fecha'].min()} → {df['fecha'].max()}")

# ================================================================
# 3. INGENIERÍA DE ATRIBUTOS SIN LEAKAGE
# ================================================================

print("\n" + "=" * 80)
print("INGENIERÍA DE ATRIBUTOS SIN DATA LEAKAGE")
print("=" * 80)

df_engineered = df.copy()
y = df_engineered[target_col]

# ------------------------------------------------
# 3.1 Rezagos
# ------------------------------------------------

for lag in [1, 2, 3, 4, 8, 12, 26, 52]:
    df_engineered[f"casos_lag_{lag}"] = y.shift(lag)

# ------------------------------------------------
# 3.2 Estadísticos móviles basados SOLO en pasado
# ------------------------------------------------

for window in [3, 5, 7, 13, 26]:
    pasado = y.shift(1)

    df_engineered[f"roll_mean_{window}"] = (
        pasado.rolling(window, min_periods=window).mean()
    )

    df_engineered[f"roll_std_{window}"] = (
        pasado.rolling(window, min_periods=window).std()
    )

    df_engineered[f"roll_max_{window}"] = (
        pasado.rolling(window, min_periods=window).max()
    )

    df_engineered[f"roll_min_{window}"] = (
        pasado.rolling(window, min_periods=window).min()
    )

# ------------------------------------------------
# 3.3 Tendencia
# ------------------------------------------------

df_engineered["tendencia"] = (
    y.shift(1)
    .rolling(13, min_periods=13)
    .mean()
)

df_engineered["tendencia_cambio"] = (
    df_engineered["tendencia"]
    - df_engineered["tendencia"].shift(4)
)

# ------------------------------------------------
# 3.4 Diferencias
# ------------------------------------------------

for lag in [1, 2, 4, 8, 12]:
    df_engineered[f"diff_lag_{lag}"] = (
        y.shift(1) - y.shift(lag + 1)
    )

df_engineered["acceleration"] = (
    (y.shift(1) - y.shift(2))
    - (y.shift(2) - y.shift(3))
)

# ------------------------------------------------
# 3.5 Cambio porcentual robusto
# ------------------------------------------------

for lag in [1, 4, 12]:
    denominator = y.shift(lag + 1).replace(0, np.nan)

    df_engineered[f"pct_change_lag_{lag}"] = (
        (y.shift(1) - y.shift(lag + 1))
        / denominator
    ) * 100

# Limitar extremos de cambios porcentuales.
for col in [
    "pct_change_lag_1",
    "pct_change_lag_4",
    "pct_change_lag_12"
]:
    if col in df_engineered.columns:
        df_engineered[col] = df_engineered[col].clip(-500, 500)

# ------------------------------------------------
# 3.6 Rangos
# ------------------------------------------------

df_engineered["range_7"] = (
    df_engineered["roll_max_7"]
    - df_engineered["roll_min_7"]
)

df_engineered["range_13"] = (
    df_engineered["roll_max_13"]
    - df_engineered["roll_min_13"]
)

# ------------------------------------------------
# 3.7 Calendario
# ------------------------------------------------

df_engineered["semana_sin"] = np.sin(
    2 * np.pi * df_engineered["semana_epi"] / 52
)

df_engineered["semana_cos"] = np.cos(
    2 * np.pi * df_engineered["semana_epi"] / 52
)

# ------------------------------------------------
# 3.8 Interacciones meteorología × pasado
#
# Solo se crean unas pocas. El script anterior creaba más
# variables de las necesarias para un conjunto de apenas
# 197 observaciones.
# ------------------------------------------------

meteo_vars = [
    col for col in predictor_cols
    if col.startswith(
        ("prec", "temp", "tmax", "tmin", "hr", "soi", "oni", "mei")
    )
]

for var in meteo_vars[:3]:
    df_engineered[f"{var}_x_casos_lag1"] = (
        df_engineered[var] * y.shift(1)
    )

# ================================================================
# 4. LIMPIEZA Y SEPARACIÓN TEMPORAL
# ================================================================

df_engineered = df_engineered.replace(
    [np.inf, -np.inf],
    np.nan
)

train_mask = (
    (df_engineered["año"] >= 2021)
    & (df_engineered["año"] <= 2025)
)

test_mask = df_engineered["año"] == TEST_YEAR

if test_mask.sum() == 0:
    raise ValueError("No se encontraron registros para 2026.")

# Eliminar únicamente filas imposibles de utilizar por los rezagos.
df_engineered = df_engineered.dropna().reset_index(drop=True)

train_mask = (
    (df_engineered["año"] >= 2021)
    & (df_engineered["año"] <= 2025)
)

test_mask = df_engineered["año"] == TEST_YEAR

new_predictor_cols = [
    col for col in df_engineered.columns
    if col not in exclude_cols + [target_col]
]

print("\n" + "=" * 80)
print("SEPARACIÓN TEMPORAL")
print("=" * 80)
print(f"TRAIN: {train_mask.sum()} registros")
print(f"TEST : {test_mask.sum()} registros")
print(f"Predictores después de ingeniería: {len(new_predictor_cols)}")

# ================================================================
# 5. MATRICES
# ================================================================

X_all = df_engineered[new_predictor_cols].copy()
y_all = df_engineered[target_col].copy()

X_train_full = X_all.loc[train_mask].copy()
y_train_full = y_all.loc[train_mask].copy()

X_test = X_all.loc[test_mask].copy()
y_test = y_all.loc[test_mask].copy()

fechas_test = df_engineered.loc[test_mask, "fecha"].values
años_test = df_engineered.loc[test_mask, "año"].values
semanas_test = df_engineered.loc[test_mask, "semana_epi"].values

# ================================================================
# 6. FUNCIONES AUXILIARES
# ================================================================

def create_walk_forward_folds(data):
    """
    Genera folds temporales.

    Ejemplo:
        Train 2022 -> Validación 2023
        Train 2022-2023 -> Validación 2024
        Train 2022-2024 -> Validación 2025

    2026 queda fuera.
    """

    years = sorted(
        data.loc[
            (data["año"] >= 2021)
            & (data["año"] <= 2025),
            "año"
        ].unique()
    )

    folds = []

    for val_year in years[1:]:
        train_years = [
            year for year in years
            if year < val_year
        ]

        train_idx = data.index[
            data["año"].isin(train_years)
        ].to_numpy()

        val_idx = data.index[
            data["año"] == val_year
        ].to_numpy()

        # Evitamos folds con entrenamiento demasiado pequeño.
        if len(train_idx) >= 40 and len(val_idx) > 0:
            folds.append(
                (
                    train_idx,
                    val_idx,
                    train_years,
                    val_year
                )
            )

    return folds


def select_features_train_only(X, y, max_features=12):
    """
    Selección de variables usando SOLO el conjunto recibido.
    Nunca debe recibir validación o test.
    """

    # Quitar variables constantes.
    nunique = X.nunique(dropna=False)

    valid_cols = nunique[
        nunique > 1
    ].index.tolist()

    X_clean = X[valid_cols].copy()

    if len(X_clean.columns) == 0:
        raise ValueError("No quedan variables después de eliminar constantes.")

    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    rf.fit(X_clean, y)

    importance = pd.DataFrame({
        "Feature": X_clean.columns,
        "Importance": rf.feature_importances_
    }).sort_values(
        "Importance",
        ascending=False
    ).reset_index(drop=True)

    n = min(
        max_features,
        len(importance)
    )

    selected = importance.head(n)["Feature"].tolist()

    return selected, importance


def create_soft_weights(y, peak_threshold, extreme_threshold):
    """
    Ponderación suave para no obligar a LightGBM a memorizar picos.

    Peso máximo limitado a 1.8.
    """

    y = np.asarray(y, dtype=float)

    weights = np.ones(len(y), dtype=float)

    weights[y >= peak_threshold] = 1.35
    weights[y >= extreme_threshold] = 1.60

    return weights


def get_fold_thresholds(y_train):
    """
    Los umbrales de pico se calculan SOLO sobre el train del fold.
    """

    y_train = np.asarray(y_train, dtype=float)

    peak = np.percentile(y_train, 80)
    extreme = np.percentile(y_train, 95)

    return peak, extreme


folds = create_walk_forward_folds(df_engineered)

print("\n" + "=" * 80)
print("FOLDS WALK-FORWARD")
print("=" * 80)

for i, (_, _, train_years, val_year) in enumerate(folds, 1):
    print(
        f"Fold {i}: "
        f"Train {train_years} → "
        f"Validación {val_year}"
    )

if len(folds) < 2:
    raise ValueError(
        "No hay suficientes folds temporales para una validación robusta."
    )

# ================================================================
# 7. OPTUNA + SELECCIÓN NESTED DE VARIABLES
# ================================================================

print("\n" + "=" * 80)
print("OPTIMIZACIÓN LIGHTGBM CONTRA SOBREAJUSTE")
print("=" * 80)


def objective(trial):

    params = {
        "objective": "regression_l1",
        "metric": "l1",
        "boosting_type": "gbdt",

        # Complejidad deliberadamente reducida.
        "num_leaves": trial.suggest_int(
            "num_leaves",
            5,
            12
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            5
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            15,
            35
        ),

        "min_split_gain": trial.suggest_float(
            "min_split_gain",
            0.05,
            0.50
        ),

        # Regularización.
        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0.5,
            5.0
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1.0,
            8.0
        ),

        # Submuestreo.
        "feature_fraction": trial.suggest_float(
            "feature_fraction",
            0.60,
            0.85
        ),

        "bagging_fraction": trial.suggest_float(
            "bagging_fraction",
            0.60,
            0.85
        ),

        "bagging_freq": trial.suggest_int(
            "bagging_freq",
            1,
            5
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.04,
            log=True
        ),

        "verbosity": -1,
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }

    fold_scores = []

    for fold_number, (
        train_idx,
        val_idx,
        train_years,
        val_year
    ) in enumerate(folds, 1):

        X_tr_all = X_all.loc[train_idx]
        y_tr = y_all.loc[train_idx]

        X_val_all = X_all.loc[val_idx]
        y_val = y_all.loc[val_idx]

        # --------------------------------------------------------
        # CORRECCIÓN CRÍTICA:
        # Selección de features SOLO sobre TRAIN del fold.
        # --------------------------------------------------------

        selected_fold_features, _ = select_features_train_only(
            X_tr_all,
            y_tr,
            MAX_SELECTED_FEATURES
        )

        X_tr = X_tr_all[selected_fold_features]
        X_val = X_val_all[selected_fold_features]

        # --------------------------------------------------------
        # Umbrales SOLO del train del fold.
        # --------------------------------------------------------

        peak_threshold_fold, extreme_threshold_fold = (
            get_fold_thresholds(y_tr)
        )

        weights_tr = create_soft_weights(
            y_tr.values,
            peak_threshold_fold,
            extreme_threshold_fold
        )

        dtrain = lgb.Dataset(
            X_tr,
            label=y_tr,
            weight=weights_tr
        )

        dval = lgb.Dataset(
            X_val,
            label=y_val,
            reference=dtrain
        )

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=MAX_BOOST_ROUND,
            valid_sets=[dval],
            valid_names=["validation"],
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=60,
                    verbose=False
                ),
                lgb.log_evaluation(0)
            ]
        )

        pred = model.predict(
            X_val,
            num_iteration=model.best_iteration
        )

        general_mae = mean_absolute_error(
            y_val,
            pred
        )

        peak_mask = (
            y_val.values >= peak_threshold_fold
        )

        if peak_mask.sum() > 0:
            peak_mae = mean_absolute_error(
                y_val.values[peak_mask],
                pred[peak_mask]
            )
        else:
            peak_mae = general_mae

        # Penalización moderada de subestimación en picos.
        if peak_mask.sum() > 0:
            underestimation = np.mean(
                np.maximum(
                    0,
                    y_val.values[peak_mask]
                    - pred[peak_mask]
                )
            )
        else:
            underestimation = 0.0

        # No damos demasiado peso al comportamiento de picos,
        # porque hay muy pocas observaciones.
        combined_score = (
            0.80 * general_mae
            + 0.15 * peak_mae
            + 0.05 * underestimation
        )

        fold_scores.append(combined_score)

    return float(np.mean(fold_scores))


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

best_params = study.best_params

print("\nMejores parámetros:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

print(
    f"\nMejor score walk-forward: "
    f"{study.best_value:.4f}"
)

# ================================================================
# 8. SELECCIÓN FINAL DE FEATURES
# ================================================================

print("\n" + "=" * 80)
print("SELECCIÓN FINAL DE FEATURES — SOLO TRAIN 2021-2025")
print("=" * 80)

selected_features_rf, feature_importance = (
    select_features_train_only(
        X_train_full,
        y_train_full,
        MAX_SELECTED_FEATURES
    )
)

print(
    f"Features después de ingeniería: "
    f"{len(new_predictor_cols)}"
)

print(
    f"Features seleccionadas: "
    f"{len(selected_features_rf)}"
)

print("\nFeatures seleccionadas:")

for i, feature in enumerate(
    selected_features_rf,
    1
):
    row = feature_importance[
        feature_importance["Feature"] == feature
    ].iloc[0]

    print(
        f"{i:2d}. "
        f"{feature:<40} "
        f"{row['Importance']:.6f}"
    )

features_file = os.path.join(
    output_dir,
    "atributos_seleccionados_rf_ant_sobreajuste.xlsx"
)

feature_importance.to_excel(
    features_file,
    index=False
)

X_train_full = X_train_full[
    selected_features_rf
].copy()

X_test = X_test[
    selected_features_rf
].copy()

# ================================================================
# 9. ESTIMACIÓN DEL NÚMERO DE ÁRBOLES
# ================================================================

print("\n" + "=" * 80)
print("ESTIMACIÓN DE ITERACIONES FINALES")
print("=" * 80)

best_iterations = []

for fold_number, (
    train_idx,
    val_idx,
    train_years,
    val_year
) in enumerate(folds, 1):

    X_tr_all = X_all.loc[train_idx]
    y_tr = y_all.loc[train_idx]

    X_val_all = X_all.loc[val_idx]
    y_val = y_all.loc[val_idx]

    # Selección independiente por fold.
    selected_fold_features, _ = select_features_train_only(
        X_tr_all,
        y_tr,
        MAX_SELECTED_FEATURES
    )

    X_tr = X_tr_all[selected_fold_features]
    X_val = X_val_all[selected_fold_features]

    peak_threshold_fold, extreme_threshold_fold = (
        get_fold_thresholds(y_tr)
    )

    weights_tr = create_soft_weights(
        y_tr.values,
        peak_threshold_fold,
        extreme_threshold_fold
    )

    dtrain = lgb.Dataset(
        X_tr,
        label=y_tr,
        weight=weights_tr
    )

    dval = lgb.Dataset(
        X_val,
        label=y_val,
        reference=dtrain
    )

    fold_params = {
        "objective": "regression_l1",
        "metric": "l1",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        **best_params
    }

    fold_model = lgb.train(
        fold_params,
        dtrain,
        num_boost_round=MAX_BOOST_ROUND,
        valid_sets=[dval],
        valid_names=["validation"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=60,
                verbose=False
            ),
            lgb.log_evaluation(0)
        ]
    )

    best_iterations.append(
        fold_model.best_iteration
    )

print(
    f"Mejores iteraciones por fold: "
    f"{best_iterations}"
)

final_num_boost_round = int(
    np.median(best_iterations)
)

# Más conservador para el modelo final.
final_num_boost_round = max(
    30,
    min(
        final_num_boost_round,
        600
    )
)

print(
    f"Número de iteraciones finales: "
    f"{final_num_boost_round}"
)

# ================================================================
# 10. ENTRENAMIENTO FINAL
# ================================================================

print("\n" + "=" * 80)
print("ENTRENAMIENTO FINAL")
print("=" * 80)

peak_threshold_final, extreme_threshold_final = (
    get_fold_thresholds(y_train_full)
)

print(
    f"Umbral pico TRAIN: "
    f"{peak_threshold_final:.3f}"
)

print(
    f"Umbral extremo TRAIN: "
    f"{extreme_threshold_final:.3f}"
)

train_weights = create_soft_weights(
    y_train_full.values,
    peak_threshold_final,
    extreme_threshold_final
)

final_params = {
    "objective": "regression_l1",
    "metric": "l1",
    "boosting_type": "gbdt",
    "verbosity": -1,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
    **best_params
}

dtrain_final = lgb.Dataset(
    X_train_full,
    label=y_train_full,
    weight=train_weights
)

model = lgb.train(
    final_params,
    dtrain_final,
    num_boost_round=final_num_boost_round,
    callbacks=[
        lgb.log_evaluation(0)
    ]
)

# ================================================================
# 11. PREDICCIONES
# ================================================================

y_train_pred = model.predict(
    X_train_full
)

y_test_pred = model.predict(
    X_test
)

# Los casos no pueden ser negativos.
y_train_pred = np.clip(
    y_train_pred,
    0,
    None
)

y_test_pred = np.clip(
    y_test_pred,
    0,
    None
)

# ================================================================
# 12. BASELINE NAIVE
# ================================================================

naive_test_pred = X_test[
    "casos_lag_1"
].values

naive_test_pred = np.clip(
    naive_test_pred,
    0,
    None
)

naive_mae = mean_absolute_error(
    y_test,
    naive_test_pred
)

naive_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        naive_test_pred
    )
)

naive_r2 = r2_score(
    y_test,
    naive_test_pred
)

# ================================================================
# 13. MÉTRICAS
# ================================================================

train_mae = mean_absolute_error(
    y_train_full,
    y_train_pred
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train_full,
        y_train_pred
    )
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

train_r2 = r2_score(
    y_train_full,
    y_train_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)

peak_mask_train = (
    y_train_full.values
    >= peak_threshold_final
)

peak_mask_test = (
    y_test.values
    >= peak_threshold_final
)

peak_mae_train = (
    mean_absolute_error(
        y_train_full.values[peak_mask_train],
        y_train_pred[peak_mask_train]
    )
    if peak_mask_train.sum() > 0
    else np.nan
)

peak_mae_test = (
    mean_absolute_error(
        y_test.values[peak_mask_test],
        y_test_pred[peak_mask_test]
    )
    if peak_mask_test.sum() > 0
    else np.nan
)

bias_train = np.mean(
    y_train_pred
    - y_train_full.values
)

bias_test = np.mean(
    y_test_pred
    - y_test.values
)

naive_peak_mae = (
    mean_absolute_error(
        y_test.values[peak_mask_test],
        naive_test_pred[peak_mask_test]
    )
    if peak_mask_test.sum() > 0
    else np.nan
)

if naive_mae > 0:
    improvement_vs_naive = (
        (naive_mae - test_mae)
        / naive_mae
    ) * 100
else:
    improvement_vs_naive = np.nan

generalization_gap_mae = (
    test_mae - train_mae
)

generalization_ratio_mae = (
    test_mae / max(train_mae, 1e-9)
)

# ================================================================
# 14. RESULTADOS
# ================================================================

print("\n" + "=" * 80)
print("RESULTADOS FINALES — VERSIÓN ANTISOBREAJUSTE")
print("=" * 80)

print(f"MAE Train       : {train_mae:.4f}")
print(f"MAE Test        : {test_mae:.4f}")
print(f"Brecha MAE      : {generalization_gap_mae:.4f}")
print(f"Ratio Test/Train: {generalization_ratio_mae:.2f}x")

print(f"\nPeak MAE Train  : {peak_mae_train:.4f}")
print(f"Peak MAE Test   : {peak_mae_test:.4f}")

print(f"\nRMSE Train      : {train_rmse:.4f}")
print(f"RMSE Test       : {test_rmse:.4f}")

print(f"\nR² Train        : {train_r2:.4f}")
print(f"R² Test         : {test_r2:.4f}")

print(f"\nBias Train      : {bias_train:.4f}")
print(f"Bias Test       : {bias_test:.4f}")

print("\nBASELINE NAIVE — y(t) = y(t-1)")
print(f"MAE Test        : {naive_mae:.4f}")
print(f"RMSE Test       : {naive_rmse:.4f}")
print(f"R² Test         : {naive_r2:.4f}")
print(f"Peak MAE Test   : {naive_peak_mae:.4f}")

print(
    f"\nMejora frente a baseline: "
    f"{improvement_vs_naive:.2f}%"
)

print("=" * 80)

# ================================================================
# 15. IMPORTANCIA LIGHTGBM
# ================================================================

lgb_importance = pd.DataFrame({
    "Feature": selected_features_rf,
    "Importance_gain": model.feature_importance(
        importance_type="gain"
    ),
    "Importance_split": model.feature_importance(
        importance_type="split"
    )
}).sort_values(
    "Importance_gain",
    ascending=False
)

# ================================================================
# 16. TABLA DE PREDICCIONES
# ================================================================

pred_df = pd.DataFrame({
    "fecha": fechas_test,
    "año": años_test,
    "semana_epi": semanas_test,
    "casos_reales": y_test.values,
    "predicciones_lightgbm": y_test_pred,
    "prediccion_naive": naive_test_pred
})

pred_df["error"] = (
    pred_df["predicciones_lightgbm"]
    - pred_df["casos_reales"]
)

pred_df["error_absoluto"] = (
    np.abs(pred_df["error"])
)

pred_df["es_pico"] = (
    pred_df["casos_reales"]
    >= peak_threshold_final
)

# ================================================================
# 17. ANÁLISIS POR RANGO
# ================================================================

max_case = max(
    200,
    int(
        np.ceil(
            y_test.max() / 50
        ) * 50
    )
)

bins = [
    -np.inf,
    5,
    10,
    20,
    50,
    100,
    200,
    max_case
]

bins_unique = sorted(
    set(bins)
)

labels_unique = [
    f"{bins_unique[i]}-{bins_unique[i+1]}"
    for i in range(
        len(bins_unique) - 1
    )
]

pred_df["rango_casos"] = pd.cut(
    pred_df["casos_reales"],
    bins=bins_unique,
    labels=labels_unique,
    include_lowest=True
)

error_analysis_rows = []

for label in pred_df[
    "rango_casos"
].dropna().unique():

    mask = (
        pred_df["rango_casos"] == label
    )

    if mask.sum() > 0:

        error_analysis_rows.append({
            "Rango": str(label),
            "Count": int(mask.sum()),
            "MAE": pred_df.loc[
                mask,
                "error_absoluto"
            ].mean(),
            "RMSE": np.sqrt(
                np.mean(
                    pred_df.loc[
                        mask,
                        "error"
                    ] ** 2
                )
            ),
            "Bias": pred_df.loc[
                mask,
                "error"
            ].mean(),
            "Max_Error": pred_df.loc[
                mask,
                "error_absoluto"
            ].max()
        })

error_analysis = pd.DataFrame(
    error_analysis_rows
)

# ================================================================
# 18. GUARDAR DATASET PROCESADO
# ================================================================

processed_file = os.path.join(
    processed_dir,
    "dataset_procesado_antisobreajuste.xlsx"
)

columns_to_save = (
    exclude_cols
    + [target_col]
    + selected_features_rf
)

columns_to_save = [
    col for col in columns_to_save
    if col in df_engineered.columns
]

df_final = df_engineered[
    columns_to_save
].copy()

df_final.to_excel(
    processed_file,
    index=False
)

# ================================================================
# 19. GUARDAR RESULTADOS EXCEL
# ================================================================

excel_file = os.path.join(
    output_dir,
    "resultados_modelo_antisobreajuste.xlsx"
)

metrics_df = pd.DataFrame({
    "Métrica": [
        "MAE",
        "RMSE",
        "R²",
        "Peak MAE 80% TRAIN",
        "Bias",
        "Brecha MAE Test-Train",
        "Ratio MAE Test/Train"
    ],
    "LightGBM Train": [
        train_mae,
        train_rmse,
        train_r2,
        peak_mae_train,
        bias_train,
        generalization_gap_mae,
        generalization_ratio_mae
    ],
    "LightGBM Test 2026": [
        test_mae,
        test_rmse,
        test_r2,
        peak_mae_test,
        bias_test,
        generalization_gap_mae,
        generalization_ratio_mae
    ],
    "Baseline Naive Test 2026": [
        naive_mae,
        naive_rmse,
        naive_r2,
        naive_peak_mae,
        np.mean(
            naive_test_pred
            - y_test.values
        ),
        np.nan,
        np.nan
    ]
})

params_df = pd.DataFrame({
    "Parámetro": list(
        best_params.keys()
    ),
    "Valor": [
        str(v)
        for v in best_params.values()
    ]
})

walk_forward_rows = []

for i, fold in enumerate(
    folds,
    1
):
    walk_forward_rows.append({
        "Fold": i,
        "Train": str(fold[2]),
        "Validacion": fold[3],
        "Best_iteration": (
            best_iterations[i - 1]
            if i - 1 < len(best_iterations)
            else np.nan
        )
    })

walk_forward_df = pd.DataFrame(
    walk_forward_rows
)

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    metrics_df.to_excel(
        writer,
        sheet_name="Metricas",
        index=False
    )

    pred_df.to_excel(
        writer,
        sheet_name="Predicciones_Test",
        index=False
    )

    feature_importance.to_excel(
        writer,
        sheet_name="Features_RF",
        index=False
    )

    lgb_importance.to_excel(
        writer,
        sheet_name="Features_LightGBM",
        index=False
    )

    params_df.to_excel(
        writer,
        sheet_name="Parametros",
        index=False
    )

    walk_forward_df.to_excel(
        writer,
        sheet_name="Walk_Forward",
        index=False
    )

    error_analysis.to_excel(
        writer,
        sheet_name="Analisis_Errores",
        index=False
    )

# ================================================================
# 20. GUARDAR MODELO
# ================================================================

model_file = os.path.join(
    output_dir,
    "modelo_lightgbm_final_antisobreajuste.txt"
)

model.save_model(
    model_file
)

# ================================================================
# 21. GRÁFICO
# ================================================================

fig, ax = plt.subplots(
    figsize=(16, 6)
)

ax.plot(
    fechas_test,
    y_test.values,
    label="Real",
    linewidth=1.5
)

ax.plot(
    fechas_test,
    y_test_pred,
    label="LightGBM",
    linewidth=1.5
)

ax.plot(
    fechas_test,
    naive_test_pred,
    label="Baseline naive",
    linewidth=1.2,
    linestyle="--"
)

ax.scatter(
    fechas_test[peak_mask_test],
    y_test.values[peak_mask_test],
    s=30,
    label="Picos definidos con TRAIN"
)

ax.set_xlabel("Fecha")
ax.set_ylabel("Casos de dengue")

ax.set_title(
    f"Predicción 2026 — "
    f"MAE LightGBM = {test_mae:.2f} | "
    f"MAE Naive = {naive_mae:.2f}"
)

ax.legend()
ax.grid(
    True,
    alpha=0.3
)

plt.xticks(
    rotation=45
)

plt.tight_layout()

plot_file = os.path.join(
    output_dir,
    "comparativa_test_2026_antisobreajuste.png"
)

plt.savefig(
    plot_file,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ================================================================
# 22. RESUMEN FINAL
# ================================================================

print("\n" + "=" * 80)
print("RESUMEN FINAL — CONTROL DE SOBREAJUSTE")
print("=" * 80)

print(
    f"✓ Registros TRAIN: "
    f"{len(X_train_full)}"
)

print(
    f"✓ Registros TEST 2026: "
    f"{len(X_test)}"
)

print(
    f"✓ Features después de ingeniería: "
    f"{len(new_predictor_cols)}"
)

print(
    f"✓ Features finales: "
    f"{len(selected_features_rf)}"
)

print(
    f"✓ MAE Train: "
    f"{train_mae:.2f}"
)

print(
    f"✓ MAE Test 2026: "
    f"{test_mae:.2f}"
)

print(
    f"✓ Brecha MAE: "
    f"{generalization_gap_mae:.2f}"
)

print(
    f"✓ Ratio Test/Train: "
    f"{generalization_ratio_mae:.2f}x"
)

print(
    f"✓ Peak MAE Test 2026: "
    f"{peak_mae_test:.2f}"
)

print(
    f"✓ R² Test 2026: "
    f"{test_r2:.4f}"
)

print(
    f"✓ MAE Baseline Naive: "
    f"{naive_mae:.2f}"
)

print(
    f"✓ Mejora frente a baseline: "
    f"{improvement_vs_naive:.2f}%"
)

print("\nArchivos generados:")

print(
    f"  - Dataset procesado: "
    f"{processed_file}"
)

print(
    f"  - Features RF: "
    f"{features_file}"
)

print(
    f"  - Resultados Excel: "
    f"{excel_file}"
)

print(
    f"  - Modelo LightGBM: "
    f"{model_file}"
)

print(
    f"  - Gráfico: "
    f"{plot_file}"
)

print("=" * 80)
print("PROCESO TERMINADO")
print("=" * 80)


c:\Users\marco\Documentos\investigacion\machine_learning_idalina\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CARGANDO DATOS
Registros originales: 270
Predictores iniciales: 168
Periodo: 2021-03-28 00:00:00 → 2026-05-31 00:00:00

INGENIERÍA DE ATRIBUTOS SIN DATA LEAKAGE

SEPARACIÓN TEMPORAL
TRAIN: 197 registros
TEST : 21 registros
Predictores después de ingeniería: 214

FOLDS WALK-FORWARD
Fold 1: Train [np.int64(2022)] → Validación 2023
Fold 2: Train [np.int64(2022), np.int64(2023)] → Validación 2024
Fold 3: Train [np.int64(2022), np.int64(2023), np.int64(2024)] → Validación 2025

OPTIMIZACIÓN LIGHTGBM CONTRA SOBREAJUSTE


Best trial: 32. Best value: 13.3073: 100%|██████████| 40/40 [01:14<00:00,  1.85s/it]



Mejores parámetros:
  num_leaves: 8
  max_depth: 5
  min_child_samples: 15
  min_split_gain: 0.32015077858214636
  reg_alpha: 0.5291903048846978
  reg_lambda: 2.7694432179583837
  feature_fraction: 0.656814525997416
  bagging_fraction: 0.8236773319700792
  bagging_freq: 5
  learning_rate: 0.025883642702122997

Mejor score walk-forward: 13.3073

SELECCIÓN FINAL DE FEATURES — SOLO TRAIN 2021-2025
Features después de ingeniería: 214
Features seleccionadas: 12

Features seleccionadas:
 1. roll_mean_3                              0.060421
 2. roll_max_3                               0.053991
 3. casos_lag_1                              0.051839
 4. temp_max_x_casos_lag1                    0.050270
 5. casos_dengue_lag_1                       0.046041
 6. temp_min_x_casos_lag1                    0.042492
 7. roll_min_3                               0.038406
 8. roll_mean_7                              0.033509
 9. casos_dengue_lag_2                       0.029472
10. roll_max_7             

dame la versión de este algoritmo, pero donde los predictores sean solo las variables meteorológicas

Ahora dame la versión de este algoritmo, pero donde los predictores sean solo las variables autoregresivas de `casos_dengue`, es decir, la versión sin los predictores meteorológicas pero si los predictores de la forma `casos_dengue_lang_i` para i de 1 hasta 12. 